# III-4 再現性：環境・乱数の種・記録・データの版（ノートブック版）

教科書の III-4 のうち、**Python で確かめる部分**（決定的な計算の設定、種のばらつきと「良い種を選ぶ」誤用、種で対にした比較と同等性の検定（TOST）、実行の記録と差分、データのハッシュ）を、上から順に実行すれば動く 1 冊にしたものです。Google Colab の GPU（CUDA）を、教科書の「借りたインスタンス」の代わりにします。Mac の Jupyter（MPS、または CPU）でも動きます。

| 節 | 内容 | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | device と決定的な設定を表示する | 4 | `repro.py` |
| 2 | 統計の動きを合成データで確かめる | 2 | `seed_stats.py --demo` |
| 3 | 決定的か（同じ種を 2 回。CPU とこの機械の GPU） | 4 | `train_repro.py`・`runlog.py diff` |
| 4 | 種のばらつきを測る（同じ種のくり返しも） | 5・12 | `run_seeds.sh`・`seed_stats.py` |
| 5 | GPU と CPU を種で対にして比べる（TOST） | 5（任意）・14 | `seed_stats.py` |
| 6 | データのハッシュ（マニフェスト・verify・diff・pack） | 6 | `datahash.py` |
| 7 | 結果を持ち帰る | 13 | ― |

- 次の「準備」のセルたちが、III-2 の `train_toy.py` と、この章の `repro.py`・`runlog.py`・`train_repro.py`・`configs/seed_small.yaml`・`run_seeds.sh`・`seed_stats.py`・`datahash.py` を、このノートブックの隣に `%%writefile` で書き出します。**中身は教科書と 1 文字も変えていません**。
- 教科書のターミナルの `uv run --locked python train_repro.py …` の代わりに、`run("train_repro.py …")` で同じ引数を渡して別のプロセスで実行します。`run_seeds.sh` は中で `uv run --locked python` を呼ぶので、このノートブックでは **`bin/uv` という小さな代わり**（`uv run --locked python` を、このノートブックの python に置き換えるだけ）を作って PATH の先頭に足します。本物の uv と `uv.lock` で環境をそろえること（手順 1・9）は、このノートブックでは確かめられません。
- Colab では `~/home-cloud` は Git の中ではないので、`runlog.py` の記録のコミット ID は `unknown` になり、「コミットしていない変更があります」では止まりません（Mac の `~/home-cloud` では、教科書のとおり先にコミットします）。Mac でこのノートブックを開いたときも、ノートブックのフォルダの上にある Git（教科書のフォルダなど）は見ないようにしてあります（0 節の設定のセルの `GIT_CEILING_DIRECTORIES`。ノートブック版で足したもの）。
- **既定は小さい設定**です（`--set data.n_train=2000 --set data.n_test=1000 --set train.epochs=4`、種 5 個）。教科書の本番（`seed_small.yaml` のまま、種 0〜9）はセルのコメントにあります。小さい設定では正解率が教科書の値（平均 0.8955 など）より低く、種のばらつきも大きくなります。`ok()` は、合成データ（種が固定）の値と、ハッシュの一致・記録の形など、設定の大きさに依らないことだけを確かめます。
- 執筆時の Mac（Apple シリコン、MPS）で、全部で 4〜6 分ほどです。Colab の CPU は遅いので、CPU の実行（3 節・5 節）が全体の時間の大半になります。
- **入れていないもの**：`pyproject.toml` と `uv lock`（手順 1）、Git へのコミット（手順 3）、Vast.ai のインスタンス（手順 8〜12）、`data_sync.sh`・`setup_uv_instance.sh`（Mac とインスタンスのシェルで使うもの。`data_sync.sh pack` の中身の `datahash.py pack` は 6 節で試します）、Google Drive からデータを持ってくる「Colab｜データ」のセル（手順 7。Drive に置いたアーカイブが要ります）、W&B（手順 15）。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

## 0. 準備

### Colab で GPU を選ぶ

1. メニューの「ランタイム」→「ランタイムのタイプを変更」を開き、ハードウェア アクセラレータで GPU（例：T4）を選んで保存します（項目の名前と、無料で選べる GPU の種類は公式ドキュメントで要確認）。
2. ランタイムを変えると、それまでに書き出したファイルと変数は消えるので、このノートブックは**最初のセルから**実行し直してください。
3. GPU を選ばなくても全部のセルは動きます（計算の場所が `cpu` になり、5 節の「GPU と CPU の比較」はとばします）。

計算の場所（device）は `repro.py` の `pick_device()` が **cuda → mps → cpu** の順に自動で選びます。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・pandas・scipy・matplotlib・pyyaml・tensorboard が最初から入っている。版は公式で要確認）
import importlib, subprocess, sys
for mod, pkg in [("torch", "torch"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                 ("yaml", "pyyaml"), ("tensorboard", "tensorboard")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, torch, platform, os
print("Python", platform.python_version(), " numpy", numpy.__version__, " torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())
print("（教科書は torch 2.14.1 で確かめた。Colab の torch の版は違うことがある。それも run_record.json に残る）")
os.makedirs("configs", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

`train_toy.py`（III-2 の 6 節）を書き出します。

In [ ]:
%%writefile train_toy.py
#!/usr/bin/env python3
"""III-2：どこでも動く練習用の学習スクリプト
合成した「モータ音」の正常／異常を、1次元CNNで分類する。
- device を自動で選ぶ（cuda → mps → cpu）。同じコードが Mac と Colab で動く
- 一定時間ごと＋エポックの終わりにチェックポイントを保存。切断されても途中から正確に再開できる
- CUDA では混合精度（AMP）を使う
- --max-hours を過ぎたら、保存して自分で終了する（課金の止め忘れ対策）
使い方:
  uv run python train_toy.py --quick --ckpt-dir runs/mac_quick              # Mac で動作確認
  python train_toy.py --ckpt-dir /content/drive/MyDrive/home-lab/runs/toy01    # Colab で本番
"""
import argparse
import contextlib
import csv
import json
import math
import os
import platform
import random
import signal
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

FS = 16000
stop_requested = False


def on_signal(signum, frame):
    global stop_requested
    stop_requested = True
    print(f"\nシグナル {signum} を受信：区切りのよいところで保存して終了します", flush=True)


# ---------------------------------------------------------------- 環境
def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS"
    return platform.processor() or platform.machine()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=5)
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def amp_ctx(enabled):
    return torch.autocast("cuda", dtype=torch.float16) if enabled else contextlib.nullcontext()


# ---------------------------------------------------------------- データ（合成）
def make_signal(rng, length, anomalous):
    """基本波 80〜120 Hz と高調波＋雑音。異常は「周期的なクリック」か「高い周波数のうなり」"""
    t = np.arange(length) / FS
    f0 = rng.uniform(80, 120)
    x = np.zeros(length)
    for k in range(1, 6):
        x += np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k
    x += rng.normal(0, 0.3, length)
    if anomalous:
        if rng.integers(2) == 0:
            period = int(FS / rng.uniform(15, 40))
            win = np.hanning(16) * rng.uniform(2, 4)
            for p in range(int(rng.integers(period)), length, period):
                seg = x[p:p + 16]
                seg += win[:len(seg)]
        else:
            fm = rng.uniform(1500, 3000)
            env = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(5, 20) * t)
            x += 0.6 * np.sin(2 * np.pi * fm * t) * env
    return (x / x.std()).astype(np.float32)


class SynthMotor(Dataset):
    """番号 i から決まる乱数で毎回同じ信号を作る（ファイルに保存しないのでディスクを使わない）"""
    def __init__(self, n, length, seed):
        self.n, self.length, self.seed = n, length, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, i])
        y = i % 2                                 # 正常と異常を半分ずつ
        x = make_signal(rng, self.length, bool(y))
        return torch.from_numpy(x).unsqueeze(0), y


# ---------------------------------------------------------------- モデル
class AudioCNN(nn.Module):
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


# ---------------------------------------------------------------- チェックポイント
def save_ckpt(path, state):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)                         # 書き込み途中で切れても壊れない


@torch.no_grad()
def evaluate(model, loader, dev, use_amp):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with amp_ctx(use_amp):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--n-val", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ckpt-dir", default="runs/toy")
    ap.add_argument("--ckpt-every-min", type=float, default=5.0)
    ap.add_argument("--max-hours", type=float, default=3.0, help="この時間を過ぎたら保存して終了")
    ap.add_argument("--quick", action="store_true", help="小さい設定で動作確認")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs = 4000, 400, 2
    return a


def main():
    a = parse()
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    set_seed(a.seed)
    dev = pick_device(a.device)
    use_amp = dev.type == "cuda" and not a.no_amp
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.ckpt_dir, exist_ok=True)
    last = os.path.join(a.ckpt_dir, "last.pt")
    metrics_path = os.path.join(a.ckpt_dir, "metrics.csv")
    t_start = time.time()

    train_ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    val_ds = SynthMotor(a.n_val, a.length, seed=a.seed + 10_000)
    pin = dev.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=a.batch * 2, num_workers=a.workers, pin_memory=pin)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, start_step = 0, 0
    run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
    resumed = False
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)   # 自分で作ったファイルなので可
        if ck.get("done"):
            print(f"この実行はすでに完了しています: {last}")
            return
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, start_step, run = ck["epoch"], ck["step"], ck["run"]
        resumed = True
        print(f"チェックポイントから再開: epoch {start_epoch} step {start_step}", flush=True)

    # 実行の記録（再開のたびに追記）
    info_path = os.path.join(a.ckpt_dir, "run_info.json")
    info = json.load(open(info_path, encoding="utf-8")) if os.path.exists(info_path) else {"sessions": []}
    info["sessions"].append({
        "started": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "git_commit": git_commit(),
        "torch": torch.__version__, "python": sys.version.split()[0],
        "device": dev.type, "device_name": device_name(dev), "amp": use_amp,
        "args": vars(a), "resumed": resumed})
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"device = {device_name(dev)}（{dev.type}）, AMP = {use_amp}, commit = {git_commit()}", flush=True)

    def state(epoch, step, run_state, done=False):
        return {"model": model.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "step": step,
                "run": run_state, "args": vars(a), "done": done,
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

    last_save = time.time()
    for epoch in range(start_epoch, a.epochs):
        first = epoch == start_epoch
        if not first:
            run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
        step = start_step if first else 0
        # エポックごとの並び順を「種＋エポック番号」で決める → 途中からでも同じ順番で再開できる
        g = torch.Generator().manual_seed(a.seed * 1000 + epoch)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        loader = DataLoader(Subset(train_ds, order[step * a.batch:]), batch_size=a.batch,
                            shuffle=False, num_workers=a.workers, pin_memory=pin)
        model.train()
        t_seg = time.time()
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(use_amp):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run["loss_sum"] += loss.item() * len(y)
            run["n"] += len(y)
            step += 1
            over_time = time.time() - t_start > a.max_hours * 3600
            if stop_requested or over_time or time.time() - last_save > a.ckpt_every_min * 60:
                run["t"] += time.time() - t_seg
                t_seg = time.time()
                save_ckpt(last, state(epoch, step, run))
                last_save = time.time()
                print(f"  チェックポイント保存: epoch {epoch} step {step}", flush=True)
                if stop_requested or over_time:
                    why = "シグナル" if stop_requested else f"--max-hours {a.max_hours}"
                    print(f"{why} により終了します。同じコマンドで再開できます。", flush=True)
                    return
        run["t"] += time.time() - t_seg
        val_acc = evaluate(model, val_loader, dev, use_amp)
        row = {"epoch": epoch, "train_loss": run["loss_sum"] / max(run["n"], 1),
               "val_acc": val_acc, "epoch_time_s": run["t"],
               "samples_per_s": run["n"] / max(run["t"], 1e-9),
               "device": dev.type, "device_name": device_name(dev),
               "resumed": resumed and first, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
        new = not os.path.exists(metrics_path)
        with open(metrics_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}  "
              f"{row['samples_per_s']:.0f} サンプル/秒", flush=True)
        save_ckpt(last, state(epoch + 1, 0, {"loss_sum": 0.0, "n": 0, "t": 0.0},
                              done=(epoch + 1 == a.epochs)))
        last_save = time.time()
    print("完了しました。", flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

`repro.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile repro.py
#!/usr/bin/env python3
"""III-4：device の選択・乱数の種・決定的な計算の設定を1か所にまとめる（Mac の MPS／CPU、クラウドの CUDA で共通）
使い方（ほかのスクリプトから）:
  from repro import pick_device, set_seed, set_determinism, seed_worker, make_generator, state_sha256
  set_determinism(True)            # ← CUDA を使う前（最初）に呼ぶ
  dev = pick_device("auto")
  set_seed(3)
動作の確認:
  uv run python repro.py           # 選ばれた device と設定を表示する
"""
import hashlib
import os
import random

import numpy as np
import torch


def pick_device(name="auto"):
    """cuda → mps → cpu の順に探す（III-1 の bench_step.py と同じ順番）"""
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed):
    """Python・NumPy・PyTorch（CPU と全部の GPU）の乱数の種をそろえる"""
    random.seed(seed)
    np.random.seed(seed % 2**32)
    torch.manual_seed(seed)          # CPU と CUDA の両方に効く（MPS も torch.manual_seed で設定される）


def seed_worker(worker_id):
    """DataLoader の worker ごとに NumPy と random の種を決める（PyTorch の公式の書き方）"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed):
    """データの並び順を決める乱数。CPU で作るので、どの device でも同じ並び順になる"""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def set_determinism(deterministic=True, tf32=False):
    """決定的な計算の設定。CUDA の最初の計算より前に呼ぶ（CUBLAS_WORKSPACE_CONFIG は cuBLAS の初期化時に読まれる）
    deterministic=True : 同じ機械・同じ版・同じ種なら、何度実行しても同じ結果を目指す（少し遅くなる）
    deterministic=False: 速さ優先（cudnn.benchmark=True）。同じ種でも実行ごとに少しずれる
    tf32=False        : Ampere 以降の GPU の TF32（float32 の行列積・畳み込みを精度を落として速くする）を使わない
    返り値: 実際に設定した内容（run_record.json に残す）
    """
    info = {"deterministic": bool(deterministic), "tf32": bool(tf32)}
    if deterministic:
        # 公式（NVIDIA cuBLAS の Results Reproducibility）の値。:16:8 はメモリが少ないが遅いことがある
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        # MPS には決定的な実装が無い演算があるので、MPS では「エラー」ではなく「警告」にする
        mps = torch.backends.mps.is_available() and not torch.cuda.is_available()
        torch.use_deterministic_algorithms(True, warn_only=mps)
        info["warn_only"] = mps
    else:
        torch.backends.cudnn.benchmark = True
        torch.use_deterministic_algorithms(False)
    # TF32 の切り替え（新しい版では別の名前の設定もある。公式ドキュメントで要確認）
    torch.set_float32_matmul_precision("high" if tf32 else "highest")
    torch.backends.cuda.matmul.allow_tf32 = bool(tf32)
    torch.backends.cudnn.allow_tf32 = bool(tf32)
    info["CUBLAS_WORKSPACE_CONFIG"] = os.environ.get("CUBLAS_WORKSPACE_CONFIG", "")
    info["cudnn_benchmark"] = torch.backends.cudnn.benchmark
    return info


def state_sha256(model):
    """モデルの重みのハッシュ（先頭16文字）。2回の実行がビット単位で同じかを調べる"""
    h = hashlib.sha256()
    for k, v in sorted(model.state_dict().items()):
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()[:16]


if __name__ == "__main__":
    info = set_determinism(True)
    dev = pick_device()
    set_seed(0)
    print("device:", dev, "/ torch", torch.__version__, "/ CUDA（ビルド）", torch.version.cuda)
    print("設定:", info)
    x = torch.randn(3, generator=make_generator(0))
    print("make_generator(0) の最初の3個:", [round(v, 4) for v in x.tolist()], "（どの機械でも同じ値になるはず）")

`runlog.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile runlog.py
#!/usr/bin/env python3
"""III-4：実行の記録（コミットID・設定・環境・データ）を run_record.json に残す。2つの記録の違いを表示する
- 設定は YAML。--set a.b=値 で一部だけ上書きできる（Hydra の上書きの書き方をまねた最小限のもの）
- 記録しないもの：環境変数の値（決まった名前だけ）、ホスト名（個人名が入ることがある）、秘密の値
使い方:
  （train_repro.py などから）from runlog import load_config, start_run, finish_run
  uv run python runlog.py show runs/seed_mps/seed000            # 1つの記録を見る
  uv run python runlog.py diff runs/seed_mps/seed000 runs/seed_cuda/seed000   # 何が違うか
"""
import argparse
import copy
import datetime
import hashlib
import json
import os
import platform
import subprocess
import sys

import yaml

# 記録してよい環境変数（値に秘密が入らないものだけ）。HF_TOKEN や WANDB_API_KEY は「名前があるか」だけ記録する
ENV_KEEP = ["CUBLAS_WORKSPACE_CONFIG", "PYTHONHASHSEED", "CUDA_VISIBLE_DEVICES", "OMP_NUM_THREADS",
            "UV_TORCH_BACKEND", "WANDB_MODE", "HF_HOME"]
ENV_SECRET_NAMES = ["HF_TOKEN", "WANDB_API_KEY", "VAST_API_KEY", "GH_TOKEN"]
CODE_SUFFIX = (".py", ".sh", ".yaml", ".yml", ".toml")   # コミットしていなければ「変更あり」とみなすファイル


# ---------------------------------------------------------------- 設定ファイル
def _parse_value(text):
    """'0.01' → 0.01、'true' → True、'[1,2]' → [1, 2]（YAML の書き方で読む）"""
    return yaml.safe_load(text)


def apply_overrides(cfg, overrides):
    """['train.epochs=5', 'model.width=32'] のような上書きを適用する。無い鍵はエラー（打ち間違いを防ぐ）"""
    cfg = copy.deepcopy(cfg)
    for item in overrides or []:
        if "=" not in item:
            raise SystemExit(f"--set の書き方が違います（鍵=値）: {item}")
        key, val = item.split("=", 1)
        node = cfg
        parts = key.split(".")
        for p in parts[:-1]:
            if not isinstance(node, dict) or p not in node:
                raise SystemExit(f"設定に {key} がありません（YAML の鍵を確認）")
            node = node[p]
        if not isinstance(node, dict) or parts[-1] not in node:
            raise SystemExit(f"設定に {key} がありません（YAML の鍵を確認）")
        node[parts[-1]] = _parse_value(val)
    return cfg


def load_config(path, overrides=None):
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return apply_overrides(cfg, overrides)


def config_hash(cfg):
    """設定の中身のハッシュ（鍵の順番や空白に左右されない）。同じ値なら同じ設定"""
    text = json.dumps(cfg, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode()).hexdigest()[:12]


# ---------------------------------------------------------------- Git・ファイル・環境
def _git(*args):
    try:
        r = subprocess.run(["git", *args], capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else None
    except (OSError, subprocess.SubprocessError):
        return None


def git_info():
    commit = _git("rev-parse", "HEAD")
    if commit is None:
        return {"commit": "unknown", "dirty": None}
    lines = (_git("-c", "core.quotepath=off", "status", "--porcelain", "--untracked-files=all") or "").splitlines()
    changed = [x for x in lines if not x.startswith("??")]
    # Git に入っていない新しいコード（import されうる）も「変更あり」とみなす。data/ runs/ は .gitignore で見えない
    untracked_code = [x[3:].strip('"') for x in lines if x.startswith("??") and x.strip('"').endswith(CODE_SUFFIX)]
    return {"commit": commit, "short": commit[:7],
            "branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
            "dirty": bool(changed or untracked_code), "changed_files": changed,
            "untracked_code": untracked_code}


def file_sha256(path, n=12):
    if not os.path.exists(path):
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()[:n]


def env_info(device=None):
    info = {"python": sys.version.split()[0], "os": platform.system(), "os_release": platform.release(),
            "machine_arch": platform.machine(), "uv_lock_sha256": file_sha256("uv.lock")}
    try:
        import numpy
        info["numpy"] = numpy.__version__
    except ImportError:
        pass
    try:
        import torch
        info["torch"] = torch.__version__
        info["torch_cuda"] = torch.version.cuda
        info["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
        info["mps_available"] = torch.backends.mps.is_available()
        if device is not None and str(device).startswith("cuda"):
            info["gpu"] = torch.cuda.get_device_name(0)
            info["gpu_capability"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
    except ImportError:
        info["torch"] = None
    info["env"] = {k: os.environ[k] for k in ENV_KEEP if k in os.environ}
    info["secrets_present"] = [k for k in ENV_SECRET_NAMES if os.environ.get(k)]   # 名前だけ。値は残さない
    return info


# ---------------------------------------------------------------- 記録
def start_run(outdir, cfg, seed, machine, device=None, extra=None, allow_dirty=False):
    """実行の最初に呼ぶ。run_record.json を書き、コミットしていない変更があれば差分も残す"""
    os.makedirs(outdir, exist_ok=True)
    g = git_info()
    if g.get("dirty"):
        if not allow_dirty:
            raise SystemExit("コミットしていない変更があります。コミットしてから実行するか、"
                             "練習なら --allow-dirty を付けてください（差分を uncommitted.diff に残します）")
        with open(os.path.join(outdir, "uncommitted.diff"), "w", encoding="utf-8") as f:
            f.write(_git("diff", "HEAD") or "")
            for p in g.get("untracked_code", []):          # 新しいファイルは中身ごと残す
                f.write(f"\n# ===== Git に入っていないファイル: {p} =====\n")
                try:
                    with open(p, encoding="utf-8", errors="replace") as src:
                        f.write(src.read())
                except OSError:
                    f.write("（読めませんでした）\n")
    rec = {"started": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
           "argv": sys.argv, "machine": machine, "seed": seed,
           "git": g, "config": cfg, "config_hash": config_hash(cfg),
           "env": env_info(device), "extra": extra or {}}
    write_json(os.path.join(outdir, "run_record.json"), rec)
    return rec


def finish_run(outdir, rec, results):
    rec = dict(rec)
    rec["finished"] = datetime.datetime.now().astimezone().isoformat(timespec="seconds")
    rec["results"] = results
    write_json(os.path.join(outdir, "run_record.json"), rec)
    return rec


def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)


# ---------------------------------------------------------------- 表示・比較
def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "."))
        else:
            out[key] = v
    return out


def load_record(path):
    if os.path.isdir(path):
        path = os.path.join(path, "run_record.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


IGNORE = ("started", "finished", "argv", "results.wall_s", "git.changed_files", "git.untracked_code")


def diff_records(a, b):
    fa, fb = flatten(a), flatten(b)
    rows = []
    for k in sorted(set(fa) | set(fb)):
        if k.startswith(IGNORE):
            continue
        if fa.get(k) != fb.get(k):
            rows.append((k, fa.get(k), fb.get(k)))
    return rows


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    s = sub.add_parser("show")
    s.add_argument("run")
    d = sub.add_parser("diff")
    d.add_argument("a")
    d.add_argument("b")
    a = ap.parse_args()
    if a.cmd == "show":
        r = load_record(a.run)
        for k, v in flatten(r).items():
            print(f"{k:32s} {v}")
    else:
        rows = diff_records(load_record(a.a), load_record(a.b))
        if not rows:
            print("違いはありません（時刻・コマンドラインを除く）")
        for k, va, vb in rows:
            print(f"{k:32s} {str(va):28s} | {vb}")
        keys = [k for k, _, _ in rows]
        important = [k for k in keys if k.startswith(("git.commit", "env.uv_lock", "config_hash", "env.torch",
                                                      "extra.data_id"))]
        if important:
            print("\n注意：コード・環境・設定・データのどれかが違います。結果の差がこれのせいか、"
                  "種や device のせいかを区別できません:", ", ".join(important))


if __name__ == "__main__":
    main()

`train_repro.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile train_repro.py
#!/usr/bin/env python3
"""III-4：種を1つ決めて小さな学習を1回行い、結果と「何で動かしたか」を全部残す
- データ（合成のモータ音）とモデルは III-2 の train_toy.py のものを使う
- 種を変えて何回も実行し（run_seeds.sh）、seed_stats.py で種のばらつき・MPS と CUDA の差を調べる
- 結果は OUT/seedNNN[_rK]/ に run_record.json・metrics.csv・final.json・tb/（TensorBoard）
使い方:
  uv run python train_repro.py --config configs/seed_small.yaml --seed 0 --out runs/seed_mps --machine mac
  uv run python train_repro.py --config configs/seed_small.yaml --seed 0 --out runs/seed_cpu --device cpu --rep 1
  uv run python train_repro.py ... --set train.epochs=5 --tracker none
"""
import argparse
import csv
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from repro import make_generator, pick_device, seed_worker, set_determinism, set_seed, state_sha256
from runlog import config_hash, finish_run, load_config, start_run, write_json
from train_toy import AudioCNN, SynthMotor


class NoisyMotor(Dataset):
    """train_toy.py の SynthMotor に雑音を足して、問題を難しくする（正解率が 1 に張り付かないように）。
    雑音も番号 i と種から決めるので、何度作っても同じデータになる"""
    def __init__(self, n, length, seed, noise):
        self.base, self.seed, self.noise = SynthMotor(n, length, seed), seed, noise

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        x, y = self.base[i]
        if self.noise > 0:
            rng = np.random.default_rng([self.seed, i, 7])
            x = x + torch.from_numpy(rng.normal(0, self.noise, x.shape[-1]).astype(np.float32))
        return x, y


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/seed_small.yaml")
    ap.add_argument("--seed", type=int, required=True, help="モデルの初期値とデータの並び順の種")
    ap.add_argument("--rep", type=int, default=0, help="同じ種のくり返しの番号（決定的かを調べる）")
    ap.add_argument("--out", required=True, help="例：runs/seed_mps")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--machine", default="mac", help="機械の名前（ホスト名は記録しないので自分で付ける）")
    ap.add_argument("--set", action="append", default=[], metavar="KEY=VALUE", help="設定の上書き")
    ap.add_argument("--tracker", choices=["tensorboard", "wandb", "none"], default="tensorboard")
    ap.add_argument("--allow-dirty", action="store_true", help="コミットしていない変更があっても実行する（練習用）")
    return ap.parse_args()


@torch.no_grad()
def accuracy(model, loader, dev):
    model.eval()
    correct = n = 0
    for x, y in loader:
        pred = model(x.to(dev)).argmax(1).cpu()
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def open_tracker(kind, outdir, cfg, name, machine, commit):
    """記録係を開く。返り値は log(dict, step) と close() を持つ小さな入れ物"""
    if kind == "tensorboard":
        from torch.utils.tensorboard import SummaryWriter
        w = SummaryWriter(log_dir=os.path.join(outdir, "tb"))

        class TB:
            def log(self, d, step):
                for k, v in d.items():
                    w.add_scalar(k, v, step)

            def close(self):
                w.flush()
                w.close()
        return TB()
    if kind == "wandb":
        # 既定は offline（キーをインスタンスに置かない）。Mac で wandb sync して送る
        os.environ.setdefault("WANDB_MODE", "offline")
        # W&B は既定でホスト名や Git の情報（リモートの URL・差分など）も送る。自分で付けた名前とコミットIDだけにする
        os.environ.setdefault("WANDB_HOST", machine)          # 実際のホスト名の代わり
        os.environ.setdefault("WANDB_DISABLE_GIT", "true")    # Git の情報を集めない（コミットIDは config に入れる）
        os.environ.setdefault("WANDB_DISABLE_CODE", "true")   # コードや差分を保存しない
        import wandb   # uv sync --group wandb で入れたときだけ使う
        run = wandb.init(project="home-cloud", name=name, config=dict(cfg, commit=commit), dir=outdir)

        class WB:
            def log(self, d, step):
                run.log(d, step=step)

            def close(self):
                run.finish()
        return WB()

    class Nothing:
        def log(self, d, step):
            pass

        def close(self):
            pass
    return Nothing()


def main():
    a = parse()
    cfg = load_config(a.config, a.set)
    rcfg = cfg["repro"]
    det = set_determinism(rcfg["deterministic"], rcfg["tf32"])   # CUDA を使う前に
    dev = pick_device(a.device)
    if rcfg.get("amp"):
        raise SystemExit("この章の比較では amp: false にします（configs/seed_small.yaml）")
    name = f"seed{a.seed:03d}" + (f"_r{a.rep}" if a.rep else "")
    outdir = os.path.join(a.out, name)
    final_path = os.path.join(outdir, "final.json")
    if os.path.exists(final_path):
        print(f"{final_path} があるので、とばします（やり直すならフォルダを消す）")
        return
    data_id = "synth-" + config_hash(cfg["data"])      # 本物のデータなら datahash.py の dataset_id を入れる
    rec = start_run(outdir, cfg, a.seed, a.machine, device=dev,
                    extra={"device": str(dev), "rep": a.rep, "determinism": det, "data_id": data_id},
                    allow_dirty=a.allow_dirty)
    print(f"{name}: device={dev} commit={rec['git'].get('short')} config={rec['config_hash']} "
          f"deterministic={det['deterministic']}", flush=True)

    d, m, t = cfg["data"], cfg["model"], cfg["train"]
    train_ds = NoisyMotor(d["n_train"], d["length"], d["seed"], d["noise"])
    val_ds = NoisyMotor(d["n_val"], d["length"], d["seed"] + 10_000, d["noise"])
    test_ds = NoisyMotor(d["n_test"], d["length"], d["seed"] + 20_000, d["noise"])
    kw = dict(num_workers=t["workers"], worker_init_fn=seed_worker)
    train_loader = DataLoader(train_ds, batch_size=t["batch"], shuffle=True,
                              generator=make_generator(a.seed), **kw)
    val_loader = DataLoader(val_ds, batch_size=256, **kw)
    test_loader = DataLoader(test_ds, batch_size=256, **kw)

    set_seed(a.seed)
    model = AudioCNN(w=m["width"])          # 初期値は CPU で作る → どの device でも同じ初期値
    init_hash = state_sha256(model)
    model = model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=t["lr"])
    # 学習率を最後に向けて0まで下げる（コサイン）。最後のエポックの値が落ち着き、種の比較がしやすくなる
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=t["epochs"] * len(train_loader))
    tracker = open_tracker(a.tracker, outdir, cfg, f"{os.path.basename(a.out)}/{name}",
                           a.machine, rec["git"].get("commit"))

    rows, step, t0 = [], 0, time.time()
    for epoch in range(t["epochs"]):
        loss_sum = n = 0
        for x, y in train_loader:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            sched.step()
            loss_sum += loss.item() * len(y)
            n += len(y)
            step += 1
            if step % 20 == 0:
                tracker.log({"train/loss": loss.item()}, step)
        val_acc = accuracy(model, val_loader, dev)
        row = {"epoch": epoch, "train_loss": loss_sum / n, "val_acc": val_acc, "elapsed_s": time.time() - t0}
        rows.append(row)
        tracker.log({"val/acc": val_acc, "train/epoch_loss": row["train_loss"]}, step)
        print(f"  epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}", flush=True)

    with open(os.path.join(outdir, "metrics.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    results = {"seed": a.seed, "rep": a.rep, "device": dev.type, "machine": a.machine,
               "val_acc": rows[-1]["val_acc"], "test_acc": accuracy(model, test_loader, dev),
               "final_loss": rows[-1]["train_loss"], "init_sha": init_hash,
               "state_sha": state_sha256(model), "wall_s": round(time.time() - t0, 2),
               "deterministic": det["deterministic"], "config_hash": rec["config_hash"],
               "commit": rec["git"].get("commit"), "uv_lock": rec["env"].get("uv_lock_sha256"),
               "torch": rec["env"].get("torch"), "data_id": data_id}
    tracker.close()
    finish_run(outdir, rec, results)
    write_json(final_path, results)            # 最後に書く（これがあれば「完了」）
    print(json.dumps({k: results[k] for k in ("val_acc", "test_acc", "state_sha", "wall_s")}), flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

`configs/seed_small.yaml`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile configs/seed_small.yaml
# III-4：種のばらつきを測るための小さな設定（Mac の MPS で1回 15〜40 秒程度の目安。機械と負荷で変わる）
# 値を変えるときは、このファイルを直してコミットするか、--set train.epochs=5 のように上書きする
name: seed_small
data:
  seed: 0          # 合成データの種。種のばらつきを測るときは固定する（変えるのはモデルの種だけ）
  n_train: 4000
  n_val: 1000
  n_test: 2000
  length: 4096
  noise: 1.5       # 足す雑音の標準偏差（信号の標準偏差＝1 に対して）。0 だと正解率がほぼ 1 に張り付く
model:
  width: 16
train:
  epochs: 3
  batch: 64
  lr: 0.0003       # 学習率の最大値（コサインで最後に 0 まで下げる）
  workers: 0       # 0 なら並び順が worker の数に左右されない。速さが要るときは 2〜4
repro:
  deterministic: true
  tf32: false      # MPS と CUDA を比べるので、どちらも float32 のまま計算する
  amp: false       # 混合精度は使わない（使うと CUDA だけ float16 が混じり、比較の条件がそろわない）

`run_seeds.sh`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile run_seeds.sh
#!/usr/bin/env bash
# III-4：種を FIRST〜LAST まで変えて train_repro.py を順に実行する（Mac でもインスタンスでも同じ）
# 使い方（~/home-cloud で）:
#   bash run_seeds.sh runs/seed_mps 0 9 --machine mac
#   bash run_seeds.sh runs/seed_cuda 0 9 --machine vast-m1
#   REP=1 bash run_seeds.sh runs/seed_cuda 0 2 --machine vast-m1     # 同じ種をもう一度（決定的かを調べる）
# 終わった種（final.json がある）はとばすので、途中で止まっても同じコマンドで続きから実行できる。
set -euo pipefail
[ $# -ge 3 ] || { sed -n '2,8p' "$0"; exit 1; }
out="$1"; first="$2"; last="$3"; shift 3
config="${CONFIG:-configs/seed_small.yaml}"
rep="${REP:-0}"
for s in $(seq "$first" "$last"); do
  uv run --locked python train_repro.py --config "$config" --seed "$s" --rep "$rep" --out "$out" "$@"
done
echo "終わりました: ${out}（seed ${first}〜${last}、rep ${rep}）"

`seed_stats.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile seed_stats.py
#!/usr/bin/env python3
"""III-4：乱数の種によるばらつきを測り、MPS と CUDA の差がそのばらつきに比べて大きいかを調べる
- 種ごとの結果（final.json）から、平均の95%区間と「次の1つの種」の95%予測区間を出す
- 「良い種を選ぶ」誤用：最良の種の値がどのくらい楽観的か（n 個の最大値の期待値）
- 同じ種のくり返し（--rep）で、決定的になっているか（重みのハッシュが一致するか）
- 2つのグループ（例：MPS と CUDA）を種で対にして、差の検定と同等性の検定（TOST）
使い方:
  uv run --locked python seed_stats.py --demo     # 合成データ（runs/demo_seeds/）で動きを確かめる。図は runs/seed_stats.png
  uv run --locked python seed_stats.py runs/seed_mps runs/seed_cuda --metric test_acc --margin 0.01 --out runs/seed_compare.png
--margin（同等とみなす差の幅）は、結果を見る前に決めて、ノートに書いておく。
"""
import argparse
import glob
import json
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import integrate, stats

plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False


# ---------------------------------------------------------------- 合成データ（--demo）
def make_demo(root, n_seeds=20, seed=10):
    """MPS と CUDA で同じ種 0〜19 を1回ずつ＋種0〜2 をもう1回（どちらも決定的：同じハッシュ）。
    真の値（seed_small.yaml を Mac の MPS と CPU で回した実測に近づけた）：平均 0.896、
    種の効果（初期値・並び順。device をまたいで共通）SD 0.012、device ごとに別れる部分 SD 0.0007、
    評価データの有限さ（種ごとに共通）val SD 0.008・test SD 0.005、CUDA − MPS の真の差 +0.0003"""
    rng = np.random.default_rng(seed)
    a = rng.normal(0, 0.012, n_seeds)
    ev = rng.normal(0, 0.008, n_seeds)
    et = rng.normal(0, 0.005, n_seeds)
    for dev, shift in (("mps", 0.0), ("cuda", 0.0003)):
        out = os.path.join(root, f"seed_{dev}")
        for s in range(n_seeds):
            g = 0.896 + shift + a[s] + rng.normal(0, 0.0007)
            val, test = g + ev[s], g + et[s]
            sha = f"{dev}{s:03d}"
            for rep in ((0, 1) if s < 3 else (0,)):
                d = os.path.join(out, f"seed{s:03d}" + (f"_r{rep}" if rep else ""))
                os.makedirs(d, exist_ok=True)
                res = {"seed": s, "rep": rep, "device": dev, "machine": "demo",
                       "val_acc": round(min(val, 1.0), 4), "test_acc": round(min(test, 1.0), 4),
                       "init_sha": f"init{s:03d}", "state_sha": sha, "deterministic": True,
                       "config_hash": "demo0config", "commit": "demo", "uv_lock": "demo",
                       "torch": "demo", "data_id": "demo"}
                with open(os.path.join(d, "final.json"), "w", encoding="utf-8") as f:
                    json.dump(res, f, ensure_ascii=False, indent=1)
    return [os.path.join(root, "seed_mps"), os.path.join(root, "seed_cuda")]


# ---------------------------------------------------------------- 読み込み
def load_runs(dirs):
    rows = []
    for d in dirs:
        for p in sorted(glob.glob(os.path.join(d, "seed*", "final.json"))):
            with open(p, encoding="utf-8") as f:
                r = json.load(f)
            r["group"] = os.path.basename(os.path.normpath(d))
            rows.append(r)
    if not rows:
        raise SystemExit("final.json が見つかりません: " + " ".join(dirs))
    return pd.DataFrame(rows)


def check_conditions(df):
    """比べてよい条件か（設定・データ・環境）を確かめる"""
    print("== 条件の確認 ==")
    for col in ("config_hash", "data_id", "commit", "uv_lock", "torch"):
        if col not in df:
            continue
        if col == "torch":   # 2.14.1 と 2.14.1+cu126 は同じ版（+ のあとは CUDA の置き場所の印）
            df = df.assign(torch=df["torch"].astype(str).str.split("+").str[0])
        per = df.groupby("group")[col].nunique()
        if (per > 1).any():
            print(f"  注意：同じグループの中で {col} が混ざっています: {per[per > 1].to_dict()}")
        vals = df.groupby("group")[col].first().to_dict()
        same = len(set(vals.values())) == 1
        mark = "同じ" if same else "違う"
        print(f"  {col:12s}: {mark}  {vals}")
        if not same and col in ("config_hash", "data_id"):
            print("    → 設定かデータが違うので、グループの差を device の差とは言えません")


# ---------------------------------------------------------------- 統計
def expected_max_z(n):
    """標準正規分布から n 個とったときの最大値の期待値 E[max Z]"""
    f = lambda x: x * n * stats.norm.pdf(x) * stats.norm.cdf(x) ** (n - 1)
    return integrate.quad(f, -10, 10)[0]


def summarize(y, label):
    n, m, s = len(y), y.mean(), y.std(ddof=1)
    tq = stats.t.ppf(0.975, n - 1)
    ci = (m - tq * s / np.sqrt(n), m + tq * s / np.sqrt(n))
    pi = (m - tq * s * np.sqrt(1 + 1 / n), m + tq * s * np.sqrt(1 + 1 / n))
    print(f"  {label}: n={n}  平均 {m:.4f}  SD {s:.4f}  最小 {y.min():.4f}  最大 {y.max():.4f}")
    print(f"    平均の95%区間        [{ci[0]:.4f}, {ci[1]:.4f}]（幅 ±{tq * s / np.sqrt(n):.4f}）")
    print(f"    次の1つの種の95%予測区間 [{pi[0]:.4f}, {pi[1]:.4f}]（幅 ±{tq * s * np.sqrt(1 + 1 / n):.4f}）")
    return {"n": n, "mean": m, "sd": s, "ci": ci, "pi": pi}


def best_seed_misuse(g, metric):
    """誤用の例：val で一番良い種を選び、その種の値を「この手法の性能」として報告する"""
    n = len(g)
    best = g.loc[g["val_acc"].idxmax()]
    em = expected_max_z(n)
    s = g[metric].std(ddof=1)
    print(f"  誤用の例「良い種を選ぶ」：n={n} 個の種の最大値は、平均より約 {em:.2f}×SD = {em * s:.4f} 高く出る（期待値）")
    print(f"    val が最良の種 {int(best['seed'])}：val_acc {best['val_acc']:.4f}（val の平均 {g['val_acc'].mean():.4f}）"
          f" → その種の test_acc {best['test_acc']:.4f}（test の平均 {g['test_acc'].mean():.4f}）")
    print("    選んだ val の値は楽観的。test では平均の方へ戻る（平均への回帰）。報告するのは全部の種の平均と区間")
    return em


def determinism_check(df):
    print("== 決定的か（同じ種のくり返しで、重みのハッシュが一致するか） ==")
    any_rep = False
    for grp, g in df.groupby("group"):
        multi = g.groupby("seed").filter(lambda x: len(x) > 1)
        if multi.empty:
            continue
        any_rep = True
        same = multi.groupby("seed")["state_sha"].nunique().eq(1)
        spread = multi.groupby("seed")["test_acc"].agg(lambda x: x.max() - x.min())
        print(f"  {grp}: くり返した種 {len(same)} 個のうち、ハッシュが一致 {int(same.sum())} 個。"
              f"test_acc の差の最大 {spread.max():.4f}")
    if not any_rep:
        print("  （くり返しがありません。--rep 1 で同じ種をもう一度実行すると調べられます）")


def paired_compare(a, b, metric, margin, alpha=0.05):
    """b − a を種で対にして比べる。TOST：差の 90% 区間が (−margin, +margin) に入れば同等"""
    m = a[["seed", metric]].merge(b[["seed", metric]], on="seed", suffixes=("_a", "_b"))
    n = len(m)
    if "init_sha" in a and "init_sha" in b:
        ii = a[["seed", "init_sha"]].merge(b[["seed", "init_sha"]], on="seed")
        same = int((ii["init_sha_x"] == ii["init_sha_y"]).sum())
        print(f"  初期値の重みのハッシュが2つのグループで一致した種：{same}/{len(ii)}（一致しなければ種の使い方が違う）")
    if n < 3:
        raise SystemExit("対にできる種が3個未満です")
    d = (m[f"{metric}_b"] - m[f"{metric}_a"]).to_numpy()
    dbar, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    df_ = n - 1
    t0 = dbar / se
    p_diff = 2 * stats.t.sf(abs(t0), df_)
    ci95 = (dbar - stats.t.ppf(0.975, df_) * se, dbar + stats.t.ppf(0.975, df_) * se)
    ci90 = (dbar - stats.t.ppf(1 - alpha, df_) * se, dbar + stats.t.ppf(1 - alpha, df_) * se)
    p_low = stats.t.sf((dbar + margin) / se, df_)        # H0: 差 ≤ −margin
    p_up = stats.t.cdf((dbar - margin) / se, df_)        # H0: 差 ≥ +margin
    p_tost = max(p_low, p_up)
    r = np.corrcoef(m[f"{metric}_a"], m[f"{metric}_b"])[0, 1]
    welch = stats.ttest_ind(m[f"{metric}_b"], m[f"{metric}_a"], equal_var=False)
    s_seed = np.sqrt((m[f"{metric}_a"].var(ddof=1) + m[f"{metric}_b"].var(ddof=1)) / 2)
    print(f"== 種で対にした比較（b − a、n={n}、指標 {metric}） ==")
    print(f"  差の平均 {dbar:+.4f}  差の SD {sd:.4f}  標準誤差 {se:.4f}  （種のばらつきの SD {s_seed:.4f}）")
    print(f"  差の検定（対応のある t）：t={t0:.2f}, 自由度 {df_}, p={p_diff:.3f}, 95%区間 [{ci95[0]:+.4f}, {ci95[1]:+.4f}]")
    print(f"  同等性の検定（TOST、幅 ±{margin}）：90%区間 [{ci90[0]:+.4f}, {ci90[1]:+.4f}], p={p_tost:.4f}"
          f" → {'同等と言える' if p_tost < alpha else '同等とは言えない（データが足りないか、差がある）'}")
    print(f"  参考：対にしない Welch の t 検定 p={welch.pvalue:.3f}。対の相関 r={r:.2f}"
          f"（種の効果が device をまたいで残る割合の目安）")
    # 対を無視して計算すると（別々の種で回したのと同じ扱い）：種のばらつきがそのまま差の誤差になる
    ya, yb = m[f"{metric}_a"].to_numpy(), m[f"{metric}_b"].to_numpy()
    va, vb = ya.var(ddof=1) / n, yb.var(ddof=1) / n
    se_u = np.sqrt(va + vb)
    df_u = (va + vb) ** 2 / (va ** 2 / (n - 1) + vb ** 2 / (n - 1))
    du = yb.mean() - ya.mean()
    tq = stats.t.ppf(1 - alpha, df_u)
    p_tost_u = max(stats.t.sf((du + margin) / se_u, df_u), stats.t.cdf((du - margin) / se_u, df_u))
    print(f"  参考：対を無視して計算すると（Welch の TOST）：90%区間 [{du - tq * se_u:+.4f}, {du + tq * se_u:+.4f}],"
          f" p={p_tost_u:.4f} → {'同等と言える' if p_tost_u < alpha else '同等とは言えない'}"
          f"（標準誤差 {se_u:.4f}。対にすると {se:.4f}）")
    if p_diff >= alpha and p_tost >= alpha:
        print("  誤用の例：「差の検定で p≥0.05 だから同じ」は誤り。同等と言うには TOST が要る（今回は言えない）")
    if p_diff < alpha and p_tost < alpha:
        print("  差は「ある」が、幅 ±margin より小さい（統計的に有意でも、実質的には同等）。どちらを言いたいかで検定を選ぶ")
    # TOST に必要な種の数の目安（真の差 0、検出力 80%）
    need = None
    for k in range(3, 1001):
        q = stats.t.ppf(1 - alpha, k - 1) + stats.t.ppf(1 - 0.2 / 2, k - 1)
        if k >= (q * sd / margin) ** 2:
            need = k
            break
    print(f"  目安：真の差が 0 のとき、この幅で80%の確率で同等と言うには、種が約 {need} 個要る（差の SD {sd:.4f} から）")
    return {"d": d, "dbar": dbar, "ci90": ci90, "ci95": ci95, "p_tost": p_tost, "p_diff": p_diff,
            "merged": m, "r": r, "need": need}


# ---------------------------------------------------------------- 図
def plot(df, groups, metric, res, margin, ems, out):
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    x0 = ax[0, 0]
    for i, grp in enumerate(groups):
        g = df[(df["group"] == grp) & (df["rep"] == 0)][metric].to_numpy()
        jit = np.random.default_rng(0).uniform(-0.12, 0.12, len(g))
        x0.plot(i + jit, g, "o", alpha=0.6)
        s = summarize_quiet(g)
        x0.errorbar(i + 0.3, s["mean"], yerr=[[s["mean"] - s["ci"][0]], [s["ci"][1] - s["mean"]]],
                    fmt="s", color="k", capsize=4)
        x0.errorbar(i + 0.4, s["mean"], yerr=[[s["mean"] - s["pi"][0]], [s["pi"][1] - s["mean"]]],
                    fmt="none", color="gray", capsize=4)
    x0.set_xticks(range(len(groups)))
    x0.set_xticklabels(groups)
    x0.set_ylabel(metric)
    x0.set_title("種ごとの結果（黒：平均の95%区間、灰：予測区間）")
    x1 = ax[0, 1]
    if res is not None:
        m = res["merged"]
        for _, r in m.iterrows():
            x1.plot([0, 1], [r[f"{metric}_a"], r[f"{metric}_b"]], "-o", color="tab:blue", alpha=0.4)
        x1.set_xticks([0, 1])
        x1.set_xticklabels(groups[:2])
        x1.set_title(f"同じ種を線で結ぶ（相関 r={res['r']:.2f}）")
    x2 = ax[1, 0]
    if res is not None:
        x2.axvspan(-margin, margin, color="tab:green", alpha=0.15, label=f"同等とみなす幅 ±{margin}")
        x2.errorbar(res["dbar"], 1, xerr=[[res["dbar"] - res["ci90"][0]], [res["ci90"][1] - res["dbar"]]],
                    fmt="o", capsize=5, label="差の90%区間（TOST）")
        x2.errorbar(res["dbar"], 0.6, xerr=[[res["dbar"] - res["ci95"][0]], [res["ci95"][1] - res["dbar"]]],
                    fmt="s", color="gray", capsize=5, label="差の95%区間")
        x2.plot(res["d"], np.full(len(res["d"]), 0.25), "|", color="k", ms=12, label="種ごとの差")
        x2.axvline(0, color="k", lw=0.8)
        x2.set_ylim(0, 1.4)
        x2.set_yticks([])
        x2.set_xlabel(f"{groups[1]} − {groups[0]}")
        x2.legend(fontsize=8, loc="upper left")
        x2.set_title("差と同等性の幅")
    x3 = ax[1, 1]
    ns = np.arange(1, 51)
    x3.plot(ns, [expected_max_z(k) for k in ns])
    for n_, e in ems.items():
        x3.plot(n_, e, "o", color="tab:red")
    x3.set_xlabel("試した種の数 n")
    x3.set_ylabel("最良の種の上ぶれ（SD の何倍か）")
    x3.set_title("「良い種を選ぶ」と楽観的になる量 E[max Z]")
    x3.grid(alpha=0.3)
    fig.tight_layout()
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    fig.savefig(out, dpi=120)
    print(f"図を保存しました: {out}")


def summarize_quiet(y):
    n, m, s = len(y), y.mean(), y.std(ddof=1)
    tq = stats.t.ppf(0.975, n - 1)
    return {"mean": m, "ci": (m - tq * s / np.sqrt(n), m + tq * s / np.sqrt(n)),
            "pi": (m - tq * s * np.sqrt(1 + 1 / n), m + tq * s * np.sqrt(1 + 1 / n))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dirs", nargs="*", help="runs/seed_mps runs/seed_cuda のように（2つ目 − 1つ目 を比べる）")
    ap.add_argument("--demo", action="store_true")
    ap.add_argument("--metric", default="test_acc")
    ap.add_argument("--margin", type=float, default=0.01, help="同等とみなす差の幅（結果を見る前に決める）")
    ap.add_argument("--out", default="runs/seed_stats.png", help="図の保存先（runs/ の中なら Git に入らない）")
    a = ap.parse_args()
    dirs = make_demo("runs/demo_seeds") if a.demo else a.dirs
    if not dirs:
        ap.error("フォルダを指定するか --demo を付けてください")
    df = load_runs(dirs)
    groups = [os.path.basename(os.path.normpath(d)) for d in dirs]
    check_conditions(df)
    print(f"== 種のばらつき（指標 {a.metric}、くり返しの rep=0 だけ使う） ==")
    ems = {}
    for grp in groups:
        g = df[(df["group"] == grp) & (df["rep"] == 0)]
        if len(g) < 2:
            print(f"  {grp}: 種が2個未満なので区間を出せません")
            continue
        summarize(g[a.metric].to_numpy(), grp)
        ems[len(g)] = best_seed_misuse(g, a.metric)
    determinism_check(df)
    res = None
    if len(groups) >= 2:
        res = paired_compare(df[(df["group"] == groups[0]) & (df["rep"] == 0)],
                             df[(df["group"] == groups[1]) & (df["rep"] == 0)], a.metric, a.margin)
    plot(df, groups, a.metric, res, a.margin, ems, a.out)


if __name__ == "__main__":
    main()

`datahash.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile datahash.py
#!/usr/bin/env python3
"""III-4：データのフォルダを「ハッシュの一覧（マニフェスト）」で確かめる。標準ライブラリだけで動く（インスタンスでも python3 で動く）
  manifest DIR          各ファイルの SHA-256 を DIR/.datahash.json に書く。大きさと更新時刻が前回と同じファイルは計算を省く（キャッシュ）
  verify DIR            マニフェストと中身が一致するか（転送のあとで）。一致しなければ終了コード1
  diff OLD.json NEW.json  2つのマニフェストの違い（増えた・消えた・変わったファイル）
  id DIR                データの版の名前（dataset_id：パスとハッシュの一覧のハッシュ、先頭12文字）
  pack DIR OUT.tar.gz   同じ中身なら毎回同じバイト列になるアーカイブを作る（Google Drive に置く用）
  make-demo DIR         練習用のダミーデータ（0.1MB×60個＝約6MB）を作る
使い方:
  uv run python datahash.py manifest data/demo_set
  python3 datahash.py verify /workspace/data/demo_set           # インスタンスで
"""
import argparse
import gzip
import hashlib
import json
import os
import random
import sys
import tarfile
import time

MANIFEST = ".datahash.json"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def list_files(root):
    """隠しファイル（.datahash.json や .DS_Store など）を除いた全ファイルを、パスの順に並べる"""
    out = []
    for d, dirs, files in os.walk(root):
        dirs[:] = sorted(x for x in dirs if not x.startswith("."))
        for name in sorted(files):
            if name.startswith("."):
                continue
            full = os.path.join(d, name)
            out.append(os.path.relpath(full, root).replace(os.sep, "/"))
    return sorted(out)


def dataset_id(files):
    h = hashlib.sha256()
    for f in sorted(files, key=lambda r: r["path"]):
        h.update(f"{f['path']}\0{f['sha256']}\n".encode())
    return h.hexdigest()[:12]


def load_manifest(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def build_manifest(root):
    old_path = os.path.join(root, MANIFEST)
    cache = {}
    if os.path.exists(old_path):
        for r in load_manifest(old_path)["files"]:
            cache[r["path"]] = r
    files, reused = [], 0
    for rel in list_files(root):
        st = os.stat(os.path.join(root, rel))
        c = cache.get(rel)
        if c and c["size"] == st.st_size and c.get("mtime_ns") == st.st_mtime_ns:
            digest = c["sha256"]
            reused += 1
        else:
            digest = sha256_file(os.path.join(root, rel))
        files.append({"path": rel, "size": st.st_size, "sha256": digest, "mtime_ns": st.st_mtime_ns})
    m = {"version": 1, "created": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "n_files": len(files),
         "total_bytes": sum(f["size"] for f in files), "dataset_id": dataset_id(files), "files": files}
    tmp = old_path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=1)
    os.replace(tmp, old_path)
    return m, reused


def diff_manifests(old, new):
    a = {r["path"]: r["sha256"] for r in old["files"]}
    b = {r["path"]: r["sha256"] for r in new["files"]}
    added = sorted(set(b) - set(a))
    removed = sorted(set(a) - set(b))
    changed = sorted(p for p in set(a) & set(b) if a[p] != b[p])
    return added, removed, changed


def verify(root, manifest_path=None):
    m = load_manifest(manifest_path or os.path.join(root, MANIFEST))
    now = {"files": [{"path": rel, "sha256": sha256_file(os.path.join(root, rel))} for rel in list_files(root)]}
    added, removed, changed = diff_manifests(m, now)
    return m, added, removed, changed


def pack(root, out):
    """ファイルの順番・時刻・持ち主・権限をそろえて tar にし、gzip の時刻も0にする → 同じ中身なら同じハッシュ"""
    with open(out, "wb") as raw:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0, compresslevel=6) as gz:
            with tarfile.open(fileobj=gz, mode="w", format=tarfile.PAX_FORMAT) as tar:
                for rel in list_files(root):
                    ti = tarfile.TarInfo(rel)
                    ti.size = os.path.getsize(os.path.join(root, rel))
                    ti.mtime, ti.uid, ti.gid, ti.uname, ti.gname, ti.mode = 0, 0, 0, "", "", 0o644
                    with open(os.path.join(root, rel), "rb") as f:
                        tar.addfile(ti, f)      # 少しずつ書くので、大きなデータでもメモリを使い切らない
    return sha256_file(out)


def make_demo(root, n=60, size=100_000, seed=0):
    rng = random.Random(seed)
    os.makedirs(root, exist_ok=True)
    for i in range(n):
        with open(os.path.join(root, f"clip_{i:03d}.bin"), "wb") as f:
            f.write(rng.randbytes(size))
    with open(os.path.join(root, "labels.csv"), "w", encoding="utf-8") as f:
        f.write("file,label\n")
        for i in range(n):
            f.write(f"clip_{i:03d}.bin,{i % 2}\n")


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    for c in ("manifest", "verify", "id", "make-demo"):
        sub.add_parser(c).add_argument("dir")
    sub.choices["verify"].add_argument("--manifest", help="別の場所のマニフェストと比べる")
    p = sub.add_parser("diff")
    p.add_argument("old")
    p.add_argument("new")
    p = sub.add_parser("pack")
    p.add_argument("dir")
    p.add_argument("out")
    a = ap.parse_args()

    if a.cmd == "make-demo":
        make_demo(a.dir)
        print(f"{a.dir} に練習用のダミーデータを作りました")
    elif a.cmd == "manifest":
        t0 = time.time()
        m, reused = build_manifest(a.dir)
        print(f"{m['n_files']} ファイル、{m['total_bytes'] / 1e6:.1f} MB、dataset_id = {m['dataset_id']}"
              f"（ハッシュを計算し直したのは {m['n_files'] - reused} 個、{time.time() - t0:.2f} 秒）")
    elif a.cmd == "id":
        print(load_manifest(os.path.join(a.dir, MANIFEST))["dataset_id"])
    elif a.cmd == "verify":
        m, added, removed, changed = verify(a.dir, a.manifest)
        for tag, lst in (("マニフェストに無い", added), ("足りない", removed), ("中身が違う", changed)):
            for p in lst[:20]:
                print(f"  {tag}: {p}")
            if len(lst) > 20:
                print(f"  {tag}: ほか {len(lst) - 20} 個")
        if added or removed or changed:
            print(f"不一致：{len(added)} 個多い・{len(removed)} 個足りない・{len(changed)} 個違う")
            sys.exit(1)
        print(f"一致：{m['n_files']} ファイル、dataset_id = {m['dataset_id']}")
    elif a.cmd == "diff":
        added, removed, changed = diff_manifests(load_manifest(a.old), load_manifest(a.new))
        for tag, lst in (("増えた", added), ("消えた", removed), ("変わった", changed)):
            for p in lst:
                print(f"{tag}: {p}")
        print(f"合計：増えた {len(added)}・消えた {len(removed)}・変わった {len(changed)}")
    elif a.cmd == "pack":
        digest = pack(a.dir, a.out)
        print(f"{a.out}  sha256 = {digest}  （{os.path.getsize(a.out) / 1e6:.1f} MB）")


if __name__ == "__main__":
    main()

In [ ]:
# uv の代わり（ノートブック版で足したもの）：「uv run --locked python …」を、このノートブックの python で実行する
import os, sys, stat, shlex
os.makedirs("bin", exist_ok=True)
with open("bin/uv", "w") as f:
    f.write("#!/bin/sh\n# ノートブック版：本物の uv ではない。run と --locked を読みとばし、python をこのノートブックの python にする\n"
            "while [ \"$1\" = run ] || [ \"$1\" = --locked ]; do shift; done\n"
            f"[ \"$1\" = python ] && shift\nexec {shlex.quote(sys.executable)} \"$@\"\n")
os.chmod("bin/uv", 0o755)
os.environ["PATH"] = os.path.abspath("bin") + os.pathsep + os.environ["PATH"]
print(open("bin/uv").read())

In [ ]:
# 設定と、run()・sh()・ok() を用意する（ノートブック版で足したもの）
import re, json, glob, shlex, shutil, subprocess
import numpy as np
import pandas as pd
from IPython.display import Image, display
sys.path.insert(0, ".")
from repro import pick_device

IN_COLAB = "google.colab" in sys.modules
# ノートブック版で足したもの：Mac でノートブックのフォルダの上にある Git（教科書のフォルダなど）を見ないため（見に行くと、そこのコミットしていない変更で runlog.py が止まる）
os.environ["GIT_CEILING_DIRECTORIES"] = os.path.dirname(os.path.abspath("."))
DEV = pick_device()
D = DEV.type                                  # "cuda"・"mps"・"cpu"
MACHINE = "colab" if IN_COLAB else "mac"       # 教科書の --machine mac・vast-m1 の代わり（ホスト名は記録しない）
SET = "--set data.n_train=2000 --set data.n_test=1000 --set train.epochs=4"   # 小さい設定
N_SEEDS = 5
# 本番（教科書の手順4・5）：SET = ""（configs/seed_small.yaml のまま）、N_SEEDS = 10。1 回 MPS で 15〜40 秒、CPU で 1.5〜3 分ほど
print(f"device {D}、machine {MACHINE}、設定の上書き「{SET or 'なし'}」、種 0〜{N_SEEDS - 1}")


def _stream(args, check, env=None):
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    rc = p.wait()
    if check:
        assert rc == 0, f"エラーで終わりました（returncode {rc}）。上の表示を見てください"
    return "".join(lines), rc


def run(cmd, check=True):
    """ターミナルの `uv run --locked python <cmd>` の代わり。別のプロセスで実行し、出力をそのまま表示して返す"""
    print("$ python", cmd, flush=True)
    return _stream([sys.executable, "-u", *shlex.split(cmd)], check)[0]


def sh(cmd, check=True):
    """ターミナルのコマンド（bash）の代わり。出力と終了コードを返す"""
    print("$", cmd, flush=True)
    return _stream(["bash", "-c", cmd], check)


def num(pattern, text):
    m = re.search(pattern, text)
    assert m, "表示に見つかりません: " + pattern
    return float(m.group(1))


def final(d):
    return json.load(open(os.path.join(d, "final.json"), encoding="utf-8"))


N_OK = 0
def ok(cond, msg):
    global N_OK
    assert cond, "★確かめ失敗：" + msg
    N_OK += 1
    print("OK", msg)

## 1. device と決定的な設定を表示する（手順 4、`repro.py`）

`set_determinism(True)` は、CUDA の最初の計算より前に呼びます（`CUBLAS_WORKSPACE_CONFIG` は cuBLAS の初期化のときに読まれるため）。MPS には決定的な実装が無い演算があるので、MPS では「エラー」ではなく「警告」にします（`warn_only`）。

`make_generator(0)` はデータの並び順を決める乱数で、CPU で作るので、どの機械でも同じ値になるはずです。

In [ ]:
out1 = run("repro.py")

In [ ]:
# 確かめ（教科書 III-4-5 の「確かめること」と III-4-7 の結果の例のうち、機械と設定の大きさに依らず成り立つことだけ）
from repro import make_generator
vals = [round(v, 4) for v in torch.randn(3, generator=make_generator(0)).tolist()]
ok(f"make_generator(0) の最初の3個: {vals}" in out1 and vals == [1.541, -0.2934, -2.1788],
   f"make_generator(0) の最初の 3 個 {vals} は、この機械でも筆者の Mac と同じ（CPU の乱数）")
ok("'deterministic': True" in out1 and "'tf32': False" in out1 and ":4096:8" in out1,
   "決定的な設定・TF32 を使わない・CUBLAS_WORKSPACE_CONFIG=:4096:8 が表示される")
ok(("'warn_only': True" in out1) == (D == "mps"), "warn_only は MPS のときだけ True（CUDA・CPU では決定的でない演算はエラー）")

## 2. 統計の動きを合成データで確かめる（手順 2、`seed_stats.py --demo`）

MPS と CUDA の 2 グループで、同じ種 0〜19 を 1 回ずつ（種 0〜2 はもう 1 回）回した「つもり」の合成データです。真の値は、平均 0.896、種の効果の SD 0.012、device ごとに別れる部分の SD 0.0007、CUDA − MPS の真の差 +0.0003 です。

- **平均の 95%区間**と、**次の 1 つの種の 95%予測区間**（$\pm t\,s\sqrt{1 + 1/n}$）の違い
- 「良い種を選ぶ」誤用：$n$ 個の最大値は平均より約 $E[\max Z]\times$SD 高く出る
- **種で対にした比較**（対応のある $t$ 検定）と**同等性の検定（TOST）**：差の 90%区間が $(-\Delta, +\Delta)$ に入れば同等。$\Delta$（`--margin`）は結果を見る前に決めます

In [ ]:
out2 = run("seed_stats.py --demo")
display(Image(filename="runs/seed_stats.png", width=900))

In [ ]:
# 確かめ（教科書 III-4-5 の「確かめること」と III-4-7 の結果の例のうち、機械と設定の大きさに依らず成り立つことだけ）
ok("seed_mps: n=20  平均 0.8907  SD 0.0102" in out2, "[手順2] MPS の平均 0.8907・SD 0.0102")
ok("[0.8859, 0.8954]" in out2 and "[0.8688, 0.9125]" in out2, "III-4-7：MPS の平均の 95%区間 [0.8859, 0.8954]・予測区間 [0.8688, 0.9125]")
ok("seed_cuda: n=20  平均 0.8914  SD 0.0102" in out2, "III-4-7：CUDA は平均 0.8914・SD 0.0102")
ok("約 1.87×SD" in out2, "III-4-7：最良の種の上ぶれは n = 20 で 1.87×SD")
ok(out2.count("ハッシュが一致 3 個") == 2, "III-4-7：くり返しのハッシュはどちらも 3/3 一致")
ok("差の平均 +0.0007  差の SD 0.0009" in out2 and "p=0.002, 95%区間 [+0.0003, +0.0011]" in out2,
   "[手順2] 対にした差の平均 +0.0007・SD 0.0009、対応のある t 検定 p=0.002（95%区間 [+0.0003, +0.0011]）")
ok("90%区間 [+0.0004, +0.0010], p=0.0000 → 同等と言える" in out2 and "統計的に有意でも、実質的には同等" in out2,
   "[手順2] TOST（Δ=0.01）の 90%区間 [+0.0004, +0.0010] で「同等と言える」、「統計的に有意でも、実質的には同等」の行")
ok("Welch の t 検定 p=0.833" in out2 and "r=1.00" in out2 and "種が約 3 個要る" in out2,
   "III-4-7：対にしない Welch の検定は p=0.833、対の相関 r=1.00、必要な種の数の目安は約 3 個")
from seed_stats import expected_max_z
ok(abs(expected_max_z(10) - 1.54) < 0.005 and abs(expected_max_z(20) - 1.87) < 0.005,
   f"E[max Z]：n = 10 で {expected_max_z(10):.3f}（手順5 の約 1.54）、n = 20 で {expected_max_z(20):.3f}")

## 3. 決定的か（手順 4、`train_repro.py`・`runlog.py diff`）

同じ種 0 を、CPU で 2 回、この機械の GPU（Colab なら CUDA、Mac なら MPS）で 2 回回します。2 回目は `--rep 1` です。

- CPU の 2 回の `state_sha`（学習後の重みのハッシュ）が一致すれば、決定的です。CUDA も `set_determinism(True)` で一致するはずです。MPS は版やモデルによって一致しないことがあり、それは誤りではありません（III-4-3-4）。
- CPU と GPU の `init_sha`（学習前の重み）は一致します。モデルを CPU で作ってから `.to(dev)` するためです。`state_sha` は一致しません（計算の順番・丸めが違う）。
- `runlog.py diff` で、2 つの記録の違いが `extra.device`・`results.*` などだけで、`git.commit`・`env.uv_lock_sha256`・`config_hash` は同じことを確かめます。

Colab の CPU だけのランタイムでは、GPU の 2 回はとばします。

In [ ]:
for d in ["runs/det_cpu", f"runs/det_{D}"]:
    shutil.rmtree(d, ignore_errors=True)
run(f"train_repro.py --seed 0 --out runs/det_cpu --device cpu --machine {MACHINE} {SET}")
run(f"train_repro.py --seed 0 --out runs/det_cpu --device cpu --machine {MACHINE} --rep 1 {SET}")
if D != "cpu":
    run(f"train_repro.py --seed 0 --out runs/det_{D} --device {D} --machine {MACHINE} {SET}")
    run(f"train_repro.py --seed 0 --out runs/det_{D} --device {D} --machine {MACHINE} --rep 1 {SET}")
    out3 = run(f"runlog.py diff runs/det_cpu/seed000 runs/det_{D}/seed000")

In [ ]:
# 確かめ（教科書 III-4-5 の「確かめること」と III-4-7 の結果の例のうち、機械と設定の大きさに依らず成り立つことだけ）
c0, c1 = final("runs/det_cpu/seed000"), final("runs/det_cpu/seed000_r1")
ok(c0["state_sha"] == c1["state_sha"] and c0["test_acc"] == c1["test_acc"], f"[手順4] CPU の 2 回の state_sha が一致する（{c0['state_sha']}）")
for f in ("run_record.json", "metrics.csv", "final.json", "tb"):
    assert os.path.exists(f"runs/det_cpu/seed000/{f}"), f
ok(True, "seed000/ に run_record.json・metrics.csv・final.json・tb/（TensorBoard）がある")
rec = json.load(open("runs/det_cpu/seed000/run_record.json", encoding="utf-8"))
ok(rec["machine"] == MACHINE and "hostname" not in json.dumps(rec) and rec["env"]["secrets_present"] == [],
   "run_record.json の machine は自分で付けた名前で、ホスト名も秘密の値も入っていない")
if D != "cpu":
    g0, g1 = final(f"runs/det_{D}/seed000"), final(f"runs/det_{D}/seed000_r1")
    print(f"{D} の 2 回の state_sha：{g0['state_sha']} と {g1['state_sha']} → {'一致' if g0['state_sha'] == g1['state_sha'] else '不一致'}（記録する）")
    if D == "cuda":
        ok(g0["state_sha"] == g1["state_sha"], "[手順12] CUDA の同じ種の 2 回の state_sha が一致する（決定的）")
    ok(c0["init_sha"] == g0["init_sha"] and c0["state_sha"] != g0["state_sha"], f"[手順4] CPU と {D} の init_sha は一致し、state_sha は違う")
    keys = [l.split()[0] for l in out3.splitlines() if l.strip() and not l.startswith(("注意", " "))]
    ok("extra.device" in keys and not any(k.startswith(("git.commit", "env.uv_lock", "config_hash", "extra.data_id")) for k in keys)
       and "注意" not in out3, "[手順4] runlog.py diff の違いは extra.device・results.* などだけで、git.commit・uv_lock・config_hash・data_id は同じ")

## 4. 種のばらつきを測る（手順 5・12、`run_seeds.sh`・`seed_stats.py`）

教科書と同じ `bash run_seeds.sh runs/seed_<device> 0 9 --machine … --device …` で種を変えて順に回し、`REP=1` で種 0〜2 をもう一度回します。`run_seeds.sh` は、終わった種（`final.json` がある）をとばすので、途中で止まっても同じコマンドで続きから実行できます。

結果を `seed_stats.py runs/seed_<device>` で集計し、平均の 95%区間・予測区間・「良い種を選ぶ」の上ぶれを見ます。小さい設定なので、正解率の値は教科書の目安（MPS の test_acc 平均 0.8955・SD 0.0125）より低くなります。

In [ ]:
GDIR = f"runs/seed_{D}"
shutil.rmtree(GDIR, ignore_errors=True)
sh(f"bash run_seeds.sh {GDIR} 0 {N_SEEDS - 1} --machine {MACHINE} --device {D} {SET}")
sh(f"REP=1 bash run_seeds.sh {GDIR} 0 2 --machine {MACHINE} --device {D} {SET}")
out_skip, _ = sh(f"bash run_seeds.sh {GDIR} 0 0 --machine {MACHINE} --device {D} {SET}")     # もう一度：とばされる
out4 = run(f"seed_stats.py {GDIR} --out {GDIR}.png")
display(Image(filename=f"{GDIR}.png", width=900))
# 本番（教科書の手順5。Mac）：bash run_seeds.sh runs/seed_mps 0 9 --machine mac --device mps（SET なし）
# 本番（手順12。インスタンス）：bash run_seeds.sh /workspace/runs/seed_cuda 0 9 --machine vast-m1

In [ ]:
# 確かめ（教科書 III-4-5 の「確かめること」と III-4-7 の結果の例のうち、機械と設定の大きさに依らず成り立つことだけ）
dirs = sorted(os.path.basename(p) for p in glob.glob(f"{GDIR}/seed*") if os.path.isdir(p))
want = [f"seed{s:03d}" for s in range(N_SEEDS)] + [f"seed{s:03d}_r1" for s in range(3)]
ok(sorted(want) == dirs, f"[手順5] {GDIR}/ に seed000〜seed{N_SEEDS - 1:03d} と seed000_r1〜seed002_r1 ができる")
ok("があるので、とばします" in out_skip, "終わった種（final.json がある）はとばす。止まっても同じコマンドで続きから")
same = sum(final(f"{GDIR}/seed{s:03d}")["state_sha"] == final(f"{GDIR}/seed{s:03d}_r1")["state_sha"] for s in range(3))
print(f"くり返しのハッシュ：{same}/3 一致（筆者の Mac の MPS・torch 2.14.1 では 3/3）")
if D in ("cuda", "cpu"):
    ok(same == 3, f"[手順12] {D} では同じ種の 2 回の state_sha が 3/3 一致する（決定的）")
from seed_stats import expected_max_z
ok(f"約 {expected_max_z(N_SEEDS):.2f}×SD" in out4, f"「良い種を選ぶ」の上ぶれは n = {N_SEEDS} で {expected_max_z(N_SEEDS):.2f}×SD（本番の n = 10 なら 1.54×SD）")
acc = np.array([final(f"{GDIR}/seed{s:03d}")["test_acc"] for s in range(N_SEEDS)])
ok(acc.max() < 0.99, f"test_acc は 0.99 に張り付いていない（最大 {acc.max():.4f}。data.noise が効いている）")

## 5. GPU と CPU を種で対にして比べる（手順 5 の任意・手順 14、`seed_stats.py`）

教科書では、Mac の MPS と Vast.ai の CUDA を種で対にして比べます（その予行演習として、Mac の CPU と MPS も比べます）。ここでは、この機械の GPU（4 節）と、同じ種・同じ設定の CPU を比べます。`--margin 0.01`（同等とみなす差の幅 $\Delta$）は、**結果を見る前に**決めてノートに書きます。

- 「条件の確認」で、config_hash・data_id・commit・uv_lock・torch が「同じ」であること
- init_sha が全部の種で一致すること（一致しなければ種の使い方が違う）
- 対にした差の SD は、種のばらつきの SD よりずっと小さいはずです（筆者の Mac の CPU と MPS では約 0.001 と 0.012）。そのため、対にすると少ない種でも TOST で「同等」と言いやすく、対を無視すると言いにくくなります。

CPU だけのランタイムでは、比べる相手が無いのでとばします。

In [ ]:
if D != "cpu":
    shutil.rmtree("runs/seed_cpu", ignore_errors=True)
    sh(f"bash run_seeds.sh runs/seed_cpu 0 {N_SEEDS - 1} --machine {MACHINE} --device cpu {SET}")
    out5 = run(f"seed_stats.py runs/seed_cpu {GDIR} --margin 0.01 --out runs/seed_cpu_{D}.png")
    display(Image(filename=f"runs/seed_cpu_{D}.png", width=900))
else:
    print("GPU が無いので、とばします")
# 本番（教科書の手順14）：seed_stats.py runs/seed_mps runs/seed_cuda --margin 0.01（Vast.ai から持ち帰った結果と）

In [ ]:
# 確かめ（教科書 III-4-5 の「確かめること」と III-4-7 の結果の例のうち、機械と設定の大きさに依らず成り立つことだけ）
if D != "cpu":
    cond = out5.split("== 種のばらつき")[0]
    ok(all(re.search(fr"{c}\s*: 同じ", cond) for c in ("config_hash", "data_id", "commit", "uv_lock", "torch")),
       "[手順14] 条件の確認で config_hash・data_id・commit・uv_lock・torch が「同じ」")
    ok(f"一致した種：{N_SEEDS}/{N_SEEDS}" in out5, f"[手順14] init_sha が {N_SEEDS}/{N_SEEDS} 一致")
    sd_d, sd_s = num(r"差の SD ([\d.]+)", out5), num(r"種のばらつきの SD ([\d.]+)", out5)
    print(f"対にした差の SD {sd_d:.4f}、種のばらつきの SD {sd_s:.4f}（筆者の Mac の CPU と MPS：約 0.0010 と 0.0128）")
    ok(sd_d < sd_s / 2, "対にした差の SD は、種のばらつきの SD よりずっと小さい（種の効果が device をまたいで残る）")
    se_p, se_u = num(r"対にすると ([\d.]+)", out5), num(r"標準誤差 ([\d.]+)。対にすると", out5)
    ok(se_p < se_u, f"対にした標準誤差 {se_p:.4f} は、対を無視した標準誤差 {se_u:.4f} より小さい")

## 6. データのハッシュ（手順 6、`datahash.py`）

練習用のダミーデータ（0.1 MB × 60 個と `labels.csv`）を作り、マニフェスト（各ファイルの SHA-256 の一覧）で版を確かめます。教科書の手順のとおり、

1. `make-demo` → `manifest` を 2 回（2 回目は大きさと更新時刻が同じなので、計算し直すのは 0 個）
2. 前のマニフェストを写しておく（`manifest` は `.datahash.json` を上書きするため）
3. ファイルを 1 つ書き換えて（`printf x >> …/clip_000.bin`）`verify` → 「中身が違う」で終了コード 1
4. `manifest` で作り直して `diff` → 「変わった：1」。`dataset_id` が書き換えの前後で変わる
5. `pack` を 2 回作ると SHA-256 が同じ（教科書の `data_sync.sh pack` の中身。順番・時刻・持ち主をそろえた tar と、時刻 0 の gzip）

Mac の `/tmp/old.json` の代わりに `data/old.json` を使います。

In [ ]:
shutil.rmtree("data", ignore_errors=True)
run("datahash.py make-demo data/demo_set")
m1 = run("datahash.py manifest data/demo_set")
m2 = run("datahash.py manifest data/demo_set")
shutil.copy("data/demo_set/.datahash.json", "data/old.json")
sh("printf x >> data/demo_set/clip_000.bin")
v_out, v_rc = sh(f"{shlex.quote(sys.executable)} datahash.py verify data/demo_set", check=False)
print("終了コード", v_rc)
m3 = run("datahash.py manifest data/demo_set")
d_out = run("datahash.py diff data/old.json data/demo_set/.datahash.json")
os.makedirs("data_packs", exist_ok=True)
ID = run("datahash.py id data/demo_set").strip()
p1 = run(f"datahash.py pack data/demo_set data_packs/demo_set-{ID}.tar.gz")
p2 = run(f"datahash.py pack data/demo_set data_packs/demo_set-{ID}_again.tar.gz")

In [ ]:
# 確かめ（教科書 III-4-5 の「確かめること」と III-4-7 の結果の例のうち、機械と設定の大きさに依らず成り立つことだけ）
ok("61 ファイル" in m1 and "計算し直したのは 0 個" in m2, "[手順6] 61 ファイル（データ 60 個と labels.csv）。2 回目の manifest は「計算し直したのは 0 個」")
ok(v_rc == 1 and "中身が違う: clip_000.bin" in v_out, "[手順6] 書き換えたあとの verify は「中身が違う」で終了コード 1")
ok("変わった: clip_000.bin" in d_out and "変わった 1" in d_out, "[手順6] diff で「変わった：1」")
id_old = json.load(open("data/old.json"))["dataset_id"]
ok(id_old != ID and f"dataset_id = {ID}" in m3, f"[手順6] dataset_id が書き換えの前後で変わる（{id_old} → {ID}）")
h1, h2 = (re.search(r"sha256 = (\w+)", p).group(1) for p in (p1, p2))
ok(h1 == h2, f"[手順6] pack を 2 回作ると SHA-256 が同じ（{h1[:16]}…）")
import gzip
total = json.load(open("data/demo_set/.datahash.json"))["total_bytes"]
gz = os.path.getsize(f"data_packs/demo_set-{ID}.tar.gz")
tar_bytes = len(gzip.open(f"data_packs/demo_set-{ID}.tar.gz").read())     # gzip する前の tar の大きさ
shrink = 1 - gz / tar_bytes
ok(0 < shrink < 0.02, f"III-4-7：gzip にしても {shrink * 100:.1f}% しか縮まない（乱数は縮まない。教科書は 0.8%）")
print(f"tar {tar_bytes / 1e6:.2f} MB → gzip {gz / 1e6:.2f} MB。縮んだのは tar の見出しと 512 バイト単位の詰め物の分で、"
      f"データそのもの（{total / 1e6:.2f} MB）と比べると {(1 - gz / total) * 100:+.1f}%（ほとんど縮まない）")

## 7. 結果を持ち帰る（手順 13 の `vast_sync.sh pull seed_cuda` の代わり）

Colab のファイルはランタイムを切ると消えるので、`runs/` を zip にしてダウンロードします（Mac で動かしているときは何もしません）。Mac では `~/home-cloud/runs/` の下に展開し、`uv run --locked python seed_stats.py runs/seed_mps runs/seed_cuda --margin 0.01` や `runlog.py diff` で Mac の結果と比べます。比べるときは、**同じ設定（`SET`）で回した結果どうし**にしてください（違うと config_hash が違うと表示され、差を device のせいとは言えません）。

TensorBoard で学習曲線を見るときは、Colab なら `%load_ext tensorboard` → `%tensorboard --logdir runs/seed_cuda`、Mac なら `uv run --locked tensorboard --logdir runs/seed_mps` です（ポートの使い方は III-4-7 の表）。

終わったら、Colab のランタイムを解放します（「ランタイム」メニューの「ランタイムを接続解除して削除」など。項目の名前は公式ドキュメントで要確認）。

In [ ]:
if IN_COLAB:
    zip_path = shutil.make_archive("p3c4_runs", "zip", root_dir=".", base_dir="runs")
    from google.colab import files
    files.download(zip_path)
    # from google.colab import runtime; runtime.unassign()   # 結果をダウンロードしてから。ランタイムを返す（III-2 のセル7）
else:
    print("Colab ではないので、持ち帰りはしません。結果は", os.path.abspath("runs"), "にあります")
print(sorted(os.listdir("runs")))
print(f"ok() の数：{N_OK}")

## おわりに

- 同じ機械・同じ版・同じ種なら、決定的な設定で重みのハッシュまで一致することを確かめました。device が違うと重みは一致しませんが、結果の差は種どうしの差よりずっと小さいはずです。それを言うには、差の検定ではなく、種で対にした同等性の検定（TOST）を使います。
- 「良い種を選ぶ」と、その値は楽観的になります。報告するのは、全部の種の平均と区間です。
- 実行ごとに、コミット ID・設定のハッシュ・環境・データの版（dataset_id）を `run_record.json` に残しました。2 つの結果を比べる前に、`runlog.py diff` と `seed_stats.py` の「条件の確認」で、比べてよい条件かを確かめます。
- 本番の設定（`SET = ""`、`N_SEEDS = 10`）にするときは、0 節の設定のセルを書き換えて最初から実行し直してください。